In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

jobs_data = pd.read_csv("jobs_data_cleaned_2026.csv")

jobs_data.shape

(5941, 34)

In [ ]:
jobs_data["role"].value_counts()

,count
role,
Data Engineer,1991
Data Scientist,1990
Data Analyst,1960


In [ ]:
jobs_data.groupby("role")["salary_midpoint"].describe()

,count,mean,std,min,25%,50%,75%,max
role,,,,,,,,
Data Analyst,1960.0,102184.272184,31926.091364,26476.01,80051.96,99251.295,119012.5925,269431.22
Data Engineer,1991.0,146075.945686,45455.989013,46918.23,115683.49,137856.440,168945.2050,477743.80
Data Scientist,1990.0,156488.789528,59577.670850,50619.51,118705.43,146931.150,180905.3450,494000.00


In [ ]:
jobs_data.groupby("role")["salary_midpoint"].skew()

,salary_midpoint
role,
Data Analyst,0.953633
Data Engineer,1.487259
Data Scientist,1.839811


In [ ]:
from scipy.stats import levene

analyst_salary = jobs_data.loc[
    jobs_data["role"] == "Data Analyst",
    "salary_midpoint"
]

engineer_salary = jobs_data.loc[
    jobs_data["role"] == "Data Engineer",
    "salary_midpoint"
]

scientist_salary = jobs_data.loc[
    jobs_data["role"] == "Data Scientist",
    "salary_midpoint"
]

levene_result = levene(
    analyst_salary,
    engineer_salary,
    scientist_salary
)

levene_result

LeveneResult(statistic=np.float64(125.08256540530253), pvalue=np.float64(6.172620842557456e-54))

In [ ]:
kruskal_result = stats.kruskal(
    analyst_salary,
    engineer_salary,
    scientist_salary
)

kruskal_result

KruskalResult(statistic=np.float64(1555.8493252624712), pvalue=np.float64(0.0))

In [ ]:
from scipy.stats import mannwhitneyu

comparisons = [
    ("Data Analyst", analyst_salary, "Data Engineer", engineer_salary),
    ("Data Analyst", analyst_salary, "Data Scientist", scientist_salary),
    ("Data Engineer", engineer_salary, "Data Scientist", scientist_salary)
]

for role1, salary1, role2, salary2 in comparisons:

    result = mannwhitneyu(
        salary1,
        salary2,
        alternative="two-sided"
    )

    adjusted_p = min(result.pvalue * 3, 1)

    print(f"{role1} vs {role2}")
    print(f"U statistic: {result.statistic:.2f}")
    print(f"Raw p-value: {result.pvalue:.6g}")
    print(f"Bonferroni adjusted p-value: {adjusted_p:.6g}")
    print()

Data Analyst vs Data Engineer
U statistic: 748933.50
Raw p-value: 1.43837e-246
Bonferroni adjusted p-value: 4.3151e-246

Data Analyst vs Data Scientist
U statistic: 711959.50
Raw p-value: 1.27924e-261
Bonferroni adjusted p-value: 3.83773e-261

Data Engineer vs Data Scientist
U statistic: 1794629.00
Raw p-value: 2.73069e-07
Bonferroni adjusted p-value: 8.19208e-07



In [ ]:
python_table = pd.crosstab(
    jobs_data["role"],
    jobs_data["python"]
)

python_table

python,False,True
role,,
Data Analyst,1927,33
Data Engineer,1811,180
Data Scientist,1938,52


In [ ]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(
    python_table
)

print("Chi-square statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)
print("\nExpected frequencies:")
print(expected)

Chi-square statistic: 149.41071108137115
p-value: 3.596468990364987e-33
Degrees of freedom: 2

Expected frequencies:
[[1872.5736408    87.4263592 ]
 [1902.19087696   88.80912304]
 [1901.23548224   88.76451776]]


In [ ]:
import numpy as np

n = python_table.to_numpy().sum()

cramers_v = np.sqrt(
    chi2 /
    (n * min(python_table.shape[0] - 1,
             python_table.shape[1] - 1))
)

print("Cramer's V:", cramers_v)

Cramer's V: 0.15858462886186628


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

skill_columns = [
    "python",
    "sql",
    "r",
    "java",
    "javascript",
    "sas",
    "tableau",
    "power_bi",
    "excel",
    "aws",
    "azure",
    "gcp",
    "databricks",
    "snowflake",
    "spark",
    "hadoop",
    "tensorflow",
    "pytorch",
    "scikit_learn",
    "git"
]

results = []

for skill in skill_columns:

    table = pd.crosstab(
        jobs_data["role"],
        jobs_data[skill]
    )

    chi2, p, dof, expected = chi2_contingency(table)

    n = table.to_numpy().sum()

    cramers_v = np.sqrt(
        chi2 /
        (
            n *
            min(
                table.shape[0] - 1,
                table.shape[1] - 1
            )
        )
    )

    min_expected = expected.min()

    results.append({
        "skill": skill,
        "chi_square": chi2,
        "df": dof,
        "p_value": p,
        "cramers_v": cramers_v,
        "min_expected": min_expected
    })

skill_results = pd.DataFrame(results)

skill_results["p_adjusted"] = multipletests(
    skill_results["p_value"],
    method="fdr_bh"
)[1]

skill_results["significant"] = (
    skill_results["p_adjusted"] < 0.05
)

skill_results["chi_square_assumption"] = np.where(
    skill_results["min_expected"] >= 5,
    "Met",
    "Check"
)

skill_results = skill_results.sort_values(
    "p_adjusted"
)

skill_results

,skill,chi_square,df,p_value,cramers_v,min_expected,p_adjusted,significant,chi_square_assumption
0,python,149.410711,2,3.596469e-33,0.158585,87.426359,7.192938e-32,True,Met
9,aws,144.609323,2,3.967203e-32,0.156016,38.599562,3.967203e-31,True,Met
10,azure,130.879163,2,3.801464e-29,0.148424,29.691971,2.534309e-28,True,Met
1,sql,120.485891,2,6.867857e-27,0.142409,102.602256,3.433929e-26,True,Met
12,databricks,119.087344,2,1.382012e-26,0.141580,32.661168,5.528048e-26,True,Met
14,spark,101.347967,2,9.830339e-23,0.130610,20.454469,3.276780e-22,True,Met
13,snowflake,93.481783,2,5.019828e-21,0.125439,34.640633,1.434237e-20,True,Met
8,excel,44.777565,2,1.890929e-10,0.086816,9.897324,4.727323e-10,True,Met
7,power_bi,35.229570,2,2.238701e-08,0.077006,20.454469,4.974891e-08,True,Met
3,java,26.410565,2,1.840851e-06,0.066674,6.928127,3.347002e-06,True,Met


In [ ]:
significant_skills = [
    "python",
    "sql",
    "java",
    "tableau",
    "power_bi",
    "excel",
    "aws",
    "azure",
    "databricks",
    "snowflake",
    "spark"
]

skill_role_percentages = []

for skill in significant_skills:

    # Percentage of postings within each role
    # where the skill was detected
    percentages = (
        jobs_data
        .groupby("role")[skill]
        .mean() * 100
    )

    skill_role_percentages.append({
        "Skill": skill,
        "Data Analyst (%)": percentages["Data Analyst"],
        "Data Engineer (%)": percentages["Data Engineer"],
        "Data Scientist (%)": percentages["Data Scientist"]
    })

skill_role_table = pd.DataFrame(skill_role_percentages)

skill_role_table = skill_role_table.round(2)

skill_role_table

,Skill,Data Analyst (%),Data Engineer (%),Data Scientist (%)
0,python,1.68,9.04,2.61
1,sql,4.95,9.24,1.51
2,java,0.00,0.90,0.15
3,tableau,1.53,0.35,0.30
4,power_bi,2.09,0.85,0.20
5,excel,1.38,0.15,0.00
6,aws,0.41,5.02,0.45
7,azure,0.20,4.07,0.25
8,databricks,0.36,4.22,0.40
9,snowflake,0.92,4.07,0.30
